## Loading the Preprocessed Epochs

In [1]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from scipy.signal import welch

mne.set_log_level('WARNING')
print(f"MNE version: {mne.__version__}")
print("All imports OK.")

MNE version: 1.12.1
All imports OK.


In [2]:
path = 'MNE-eegbci-data/files/eegmmidb/1.0.0/S001'
raw = mne.io.read_raw_edf(f'{path}/S001R06.edf', preload=True)
raw.plot()

C:\Users\borae\anaconda3\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


<MNEBrowseFigure size 800x800 with 4 Axes>

In [3]:



# Strip trailing dots from channel names so they match the 10-20 standard
# (PhysioNet pads names to 4 chars with dots, e.g. "Fc5." → "FC5", "C3.." → "C3")
# strip the channel names from their dots and make them upper case so it's standardized
raw.rename_channels({ch: ch.rstrip('.').upper() for ch in raw.ch_names})

# Apply standard 10-20 montage to the scalp.
montage = mne.channels.make_standard_montage('standard_1005')
raw.set_montage(montage, on_missing='warn')

# ─────────────────────────────────────────
# STEP 2 — Explore basic info
# ─────────────────────────────────────────
print("=== BASIC INFO ===")
print(raw.info)

print("\n=== CHANNEL NAMES ===")
print(raw.ch_names)

print("\n=== KEY NUMBERS ===")
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Number of channels: {len(raw.ch_names)}")
print(f"Duration: {raw.times[-1]:.1f} seconds")
print(f"Number of timepoints: {raw.n_times}")

# ─────────────────────────────────────────
# STEP 3 — Plot the raw signal
# ─────────────────────────────────────────
raw.plot(
    duration=10,
    n_channels=20,
    scalings='auto',
    title='Raw EEG — S001R01 (eyes open baseline)'
)
plt.show()

# ─────────────────────────────────────────
# STEP 4 — Plot sensor locations on head map
# ─────────────────────────────────────────
raw.plot_sensors(show_names=True)
plt.show()

# ─────────────────────────────────────────
# STEP 5 — Plot the power spectrum
# ─────────────────────────────────────────
raw.compute_psd(fmax=60).plot()
plt.show()



C:\Users\borae\AppData\Local\Temp\ipykernel_23508\504239763.py:8: RuntimeWarning: DigMontage is only a subset of info. There are 12 channel positions not present in the DigMontage. The channels missing from the montage are:

['FCZ', 'CZ', 'CPZ', 'FP1', 'FPZ', 'FP2', 'AFZ', 'FZ', 'PZ', 'POZ', 'OZ', 'IZ'].

Consider using inst.rename_channels to match the montage nomenclature, or inst.set_channel_types if these are not EEG channels, or use the on_missing parameter if the channel positions are allowed to be unknown in your analyses.
  raw.set_montage(montage, on_missing='warn')


=== BASIC INFO ===
<Info | 9 non-empty values
 bads: []
 ch_names: FC5, FC3, FC1, FCZ, FC2, FC4, FC6, C5, C3, C1, CZ, C2, C4, C6, ...
 chs: 64 EEG
 custom_ref_applied: False
 dig: 55 items (3 Cardinal, 52 EEG)
 highpass: 0.0 Hz
 lowpass: 80.0 Hz
 meas_date: 2009-08-12 16:15:00 UTC
 nchan: 64
 projs: []
 sfreq: 160.0 Hz
 subject_info: <subject_info | his_id: X, sex: 0, last_name: X>
>

=== CHANNEL NAMES ===
['FC5', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPZ', 'CP2', 'CP4', 'CP6', 'FP1', 'FPZ', 'FP2', 'AF7', 'AF3', 'AFZ', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'FZ', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FT8', 'T7', 'T8', 'T9', 'T10', 'TP7', 'TP8', 'P7', 'P5', 'P3', 'P1', 'PZ', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'POZ', 'PO4', 'PO8', 'O1', 'OZ', 'O2', 'IZ']

=== KEY NUMBERS ===
Sampling rate: 160.0 Hz
Number of channels: 64
Duration: 125.0 seconds
Number of timepoints: 20000


C:\Users\borae\anaconda3\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)
C:\Users\borae\AppData\Local\Temp\ipykernel_23508\504239763.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\borae\AppData\Local\Temp\ipykernel_23508\504239763.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\borae\anaconda3\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)
C:\Users\borae\AppData\Local\Temp\ipykernel_23508\504239763.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
epochs = mne.read_epochs('eeg_epochs_preprocessed-epo.fif')

print(f'Total epochs: {len(epochs)}')
print(f'Channels: {epochs.ch_names}')
print(f'Sampling rate: {epochs.info['sfreq']}Hz')
print(f'Window: {epochs.tmin} to {epochs.tmax} s\n')
print(f'Event type: {epochs.event_id}')

try:

    epochs_left = epochs['T1']
    epochs_right = epochs['T2']

except KeyError:

    keys = list(epochs.event_id.keys())
    epochs_left = epochs[keys[0]]
    epochs_right = epochs[keys[1]]
    print(f"Using event labels: {keys[0]} / {keys[1]}")

print(f"\nLeft hand epochs:  {len(epochs_left)}")
print(f"Right hand epochs: {len(epochs_right)}")


Total epochs: 34
Channels: ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'Fp1', 'Fpz', 'Fp2', 'AF7', 'AF3', 'AFz', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FT8', 'T7', 'T8', 'T9', 'T10', 'TP7', 'TP8', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'O1', 'Oz', 'O2', 'Iz']
Sampling rate: 160.0Hz
Window: -1.0 to 4.0 s

Event type: {'T0': 1, 'T1': 2, 'T2': 3}

Left hand epochs:  5
Right hand epochs: 10


In [5]:
BANDS = {
    'Delta': (0.5, 4),    # deep sleep, unconscious processing
    'Theta': (4,   8),    # drowsiness, memory encoding
    'Alpha': (8,  13),    # relaxed wakefulness, motor cortex idle rhythm
    'Beta':  (13, 30),    # active thinking, motor control, motor imagery
    'Gamma': (30, 45),    # high-level processing, attention
}

FS = epochs.info['sfreq']


def band_power(signal_1d, fs, fmin, fmax, nperseg=128):


    """
    Compute average power spectral density (PSD) using Welch's method.
    """

    freqs, psd = welch(signal_1d, fs=fs, nperseg=min(nperseg, len(signal_1d)))
    idx = np.logical_and(freqs>=fmin, freqs<=fmax)
    return psd[idx].mean() if idx.any() else 0.0

print('Frequency bands defined.')
for band, (fmin, fmax) in BANDS.items():

    print(f' {band:<6} {fmin:>4.1f} - {fmax:<4.1f} Hz')

Frequency bands defined.
 Delta   0.5 - 4.0  Hz
 Theta   4.0 - 8.0  Hz
 Alpha   8.0 - 13.0 Hz
 Beta   13.0 - 30.0 Hz
 Gamma  30.0 - 45.0 Hz


## Compute Band Power per Epochs

For every epoch, compute the average power in each of the five bands at three key motor cortex channels: C3 (left), C4 (right), Cz (center).

In [6]:
CHANNELS = ['C3', 'C4', 'Cz']


def compute_band_powers_for_epochs(epochs_obj, channels, bands, fs):

    """
    returns a DataFrame, one row for each epoch and a column for each (channel, band) pair.
    """

    data = epochs_obj.get_data()
    rows = []

    # data consist of trials
    for trial in data:

        row = {}

        # for every channel within the channel_list in the parameter
        for ch_name in channels:

            # find the index of the channel
            ch_idx = epochs_obj.ch_names.index(ch_name)
            
            # take the signal from that channel of the trial
            signal = trial[ch_idx]

            for bandName, (fmin, fmax) in bands.items():

                # calculate the band power for the epoch
                row[f'{bandName}_{ch_name}'] = band_power(signal, fs, fmin, fmax)

        
        rows.append(row)

    return pd.DataFrame(rows)

    
print("Computing band powers for left hand trials...")
df_left  = compute_band_powers_for_epochs(epochs_left,  CHANNELS, BANDS, FS)
df_left['condition'] = 'left'

print("Computing band powers for right hand trials...")
df_right = compute_band_powers_for_epochs(epochs_right, CHANNELS, BANDS, FS)
df_right['condition'] = 'right'

print(f"\nLeft hand:  {df_left.shape[0]} trials, {df_left.shape[1]-1} features")
print(f"Right hand: {df_right.shape[0]} trials, {df_right.shape[1]-1} features")
print()
print(df_left.head())

Computing band powers for left hand trials...
Computing band powers for right hand trials...

Left hand:  5 trials, 15 features
Right hand: 10 trials, 15 features

       Delta_C3      Theta_C3      Alpha_C3       Beta_C3      Gamma_C3  \
0  7.472286e-12  5.699253e-12  5.882997e-12  2.507901e-12  5.534266e-13   
1  1.782570e-11  6.204403e-12  3.799895e-12  2.055386e-12  5.058680e-13   
2  1.538094e-11  7.253032e-12  6.829085e-12  2.392908e-12  5.068315e-13   
3  5.982051e-12  7.287618e-12  5.928193e-12  2.274954e-12  5.567202e-13   
4  9.801458e-12  6.824852e-12  9.462471e-12  2.205166e-12  6.533298e-13   

       Delta_C4      Theta_C4      Alpha_C4       Beta_C4      Gamma_C4  \
0  7.799117e-12  5.020985e-12  5.423693e-12  2.911407e-12  6.806559e-13   
1  1.802308e-11  5.660054e-12  4.044358e-12  2.476567e-12  5.452076e-13   
2  1.516034e-11  7.880858e-12  6.840793e-12  2.472690e-12  5.035808e-13   
3  7.116673e-12  6.923766e-12  3.598888e-12  2.831499e-12  5.875768e-13   
4  1.14114

## Event-Related Desynchronization - ERD Analysis

Compare the mean alpha power between channels C3 and C4. 

Expected: 

Right hand imagery → suppresses alpha at C3 (left motor cortex, contralateral)

Left hand imagery → suppresses alpha at C4 (right motor cortex, contralateral)

This crossed pattern is what makes motor imagery BCI possible — the brain signal is spatially specific to which hand is being imagined.

In [7]:
def compute_erd(df_a, df_b, band, channel):
    
    """
    ERD (%) = (power_a - power_b) / power_a * 100
    Positive ERD means power_b is suppressed relative to power_a.
    """

    # same format with what we printed above.
    col = f'{band}_{channel}'
    
    power_a = df_a[col].mean()
    power_b = df_b[col].mean()

    # why divide by power_a?
    erd = (power_a - power_b) / power_a * 100 if power_a > 0 else 0
    return erd, power_a, power_b

print("=== ERD Analysis — Alpha Band ===\n")

for ch in ['C3', 'C4']:
    erd, p_left, p_right = compute_erd(df_left, df_right, 'Alpha', ch)
    print(f"{ch}:")
    print(f"  Left hand alpha power:  {p_left:.4f}")
    print(f"  Right hand alpha power: {p_right:.4f}")
    print(f"  ERD (left -> right):    {erd:+.1f}%")
    print()

print("Expected pattern:")
#electrode on the left.
print("  C3 should show POSITIVE ERD (alpha drops) during RIGHT hand imagery")
#electrode on the right.
print("  C4 should show POSITIVE ERD (alpha drops) during LEFT hand imagery")
print("  This crossed/contralateral pattern confirms real motor cortex activity.")
    

=== ERD Analysis — Alpha Band ===

C3:
  Left hand alpha power:  0.0000
  Right hand alpha power: 0.0000
  ERD (left -> right):    +2.5%

C4:
  Left hand alpha power:  0.0000
  Right hand alpha power: 0.0000
  ERD (left -> right):    +1.5%

Expected pattern:
  C3 should show POSITIVE ERD (alpha drops) during RIGHT hand imagery
  C4 should show POSITIVE ERD (alpha drops) during LEFT hand imagery
  This crossed/contralateral pattern confirms real motor cortex activity.


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Event-Related Desynchronization — Alpha Band (8-13 Hz)',
             fontsize=13, fontweight='bold')

for ax, ch in zip(axes, ['C3', 'C4']):
    col = f'Alpha_{ch}'
    left_vals  = df_left[col].values
    right_vals = df_right[col].values
    
    means = [left_vals.mean(), right_vals.mean()]
    sems  = [left_vals.std()/np.sqrt(len(left_vals)),
             right_vals.std()/np.sqrt(len(right_vals))]
    
    colors = ['#1E3A5F', '#9B1B3D']
    bars = ax.bar(['Left hand\nimagery', 'Right hand\nimagery'],
                  means, yerr=sems, capsize=8,
                  color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    
    erd, _, _ = compute_erd(df_left, df_right, 'Alpha', ch)
    ax.set_title(f'{ch} channel  (ERD: {erd:+.1f}%)', fontsize=11)
    ax.set_ylabel('Alpha power (μV²/Hz)', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('eeg_erd_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eeg_erd_analysis.png")

Saved: eeg_erd_analysis.png


C:\Users\borae\AppData\Local\Temp\ipykernel_23508\3395464556.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Power Spectrum Comparison

Display the full frequency spectrum for both left and right hands at C3 channel. We will analyze which frequencies differ the most.

In [9]:
fig, ax = plt.subplots(figsize=(11, 5))

ch_idx = epochs.ch_names.index('C3')

left_data  = epochs_left.get_data()[:, ch_idx, :]
right_data = epochs_right.get_data()[:, ch_idx, :]

# Average PSD across all trials
freqs_l, psd_left  = welch(left_data,  fs=FS, nperseg=128, axis=1)
freqs_r, psd_right = welch(right_data, fs=FS, nperseg=128, axis=1)

psd_left_mean  = psd_left.mean(axis=0)
psd_right_mean = psd_right.mean(axis=0)

ax.semilogy(freqs_l, psd_left_mean,  color='#1E3A5F', linewidth=2, label='Left hand imagery')
ax.semilogy(freqs_r, psd_right_mean, color='#9B1B3D', linewidth=2, label='Right hand imagery')

for band_name, (fmin, fmax) in BANDS.items():
    ax.axvspan(fmin, fmax, alpha=0.05, color='gray')
    ax.text((fmin+fmax)/2, ax.get_ylim()[1]*0.7, band_name,
            ha='center', fontsize=8, color='gray', rotation=0)
    ax.axvline(fmax,color='darkblue',linewidth=0.5,linestyle='--')

    if band_name != 'Gamma':
        
        ax.text(fmax, 0.01, f'{fmax} Hz',
                ha='center', va='bottom', fontsize=7, color='red', rotation=30,
                transform=ax.get_xaxis_transform())

    
ax.set_xlabel('Frequency (Hz)', fontsize=10)
ax.set_ylabel('Power (μV²/Hz)', fontsize=10)
ax.set_title('Average Power Spectrum — C3 Channel, All Trials', fontsize=12)
ax.set_xlim(0, 45)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('eeg_spectrum_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eeg_spectrum_comparison.png")

Saved: eeg_spectrum_comparison.png


C:\Users\borae\AppData\Local\Temp\ipykernel_23508\2784826073.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## All Bands, Channels Heatmap

A summary of showing ERD values across every channel and frequency. This will tell us which channel-band pair carries the most discriminative/significant information.

In [10]:
erd_matrix = np.zeros((len(BANDS),len(CHANNELS)))

for i, band in enumerate(BANDS.keys()):

    for j, ch in enumerate(CHANNELS):

        erd, _, _ = compute_erd(df_left, df_right, band, ch)
        erd_matrix[i,j] = erd

fig, ax = plt.subplots(figsize=(7,5))
im = ax.imshow(erd_matrix, cmap='RdBu_r', aspect='auto', vmin=-30, vmax=30)


ax.set_xticks(range(len(CHANNELS)))
ax.set_xticklabels(CHANNELS, fontsize=10)
ax.set_yticks(range(len(BANDS.keys())))
ax.set_yticklabels(BANDS.keys(), fontsize=10)
ax.set_title('ERD (%) — Left vs Right Hand Imagery\nAll Bands x All Channels',
             fontsize=11)

for i in range(len(BANDS)):

    for j in range(len(CHANNELS)):

        text_color=('white' if abs(erd_matrix[i,j]) > 15 else 'black')
        ax.text(j, i, f'{erd_matrix[i, j]:+.1f}%',
                ha='center', va='center', color=text_color, fontsize=9)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('ERD (%)', fontsize=10)

plt.tight_layout()
plt.savefig('eeg_erd_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eeg_erd_heatmap.png")

# Find the strongest feature
max_idx = np.unravel_index(np.argmax(np.abs(erd_matrix)), erd_matrix.shape)
best_band = list(BANDS.keys())[max_idx[0]]
best_ch   = CHANNELS[max_idx[1]]
print(f"\nStrongest discriminative feature: {best_band} band at {best_ch}")
print(f"ERD: {erd_matrix[max_idx]:+.1f}%")

Saved: eeg_erd_heatmap.png

Strongest discriminative feature: Delta band at C3
ERD: -20.6%


C:\Users\borae\AppData\Local\Temp\ipykernel_23508\3907598380.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Full Feature Matrix

In [11]:
df_all = pd.concat([df_left, df_right], ignore_index=True)

# Encode labels: 0 = left, 1 = right
df_all['label'] = (df_all['condition'] == 'right').astype(int)

# list of every column name except condition and label (measurement columns)
feature_cols = [c for c in df_all.columns if c not in ('condition', 'label')]

print("=== Feature matrix ===")
print(f"Total trials:  {len(df_all)}")
print(f"Features:      {len(feature_cols)}")
print(f"  ({len(BANDS)} bands x {len(CHANNELS)} channels)")
print()
print(f"Class balance:")
print(f"  Left (0):  {(df_all['label']==0).sum()} trials")
print(f"  Right (1): {(df_all['label']==1).sum()} trials")
print()
print(df_all[feature_cols + ['label']])

=== Feature matrix ===
Total trials:  15
Features:      15
  (5 bands x 3 channels)

Class balance:
  Left (0):  5 trials
  Right (1): 10 trials

        Delta_C3      Theta_C3      Alpha_C3       Beta_C3      Gamma_C3  \
0   7.472286e-12  5.699253e-12  5.882997e-12  2.507901e-12  5.534266e-13   
1   1.782570e-11  6.204403e-12  3.799895e-12  2.055386e-12  5.058680e-13   
2   1.538094e-11  7.253032e-12  6.829085e-12  2.392908e-12  5.068315e-13   
3   5.982051e-12  7.287618e-12  5.928193e-12  2.274954e-12  5.567202e-13   
4   9.801458e-12  6.824852e-12  9.462471e-12  2.205166e-12  6.533298e-13   
5   1.173624e-11  7.533495e-12  5.105338e-12  2.252548e-12  5.240529e-13   
6   2.223332e-11  6.830999e-12  6.976035e-12  2.270401e-12  8.832512e-13   
7   1.133028e-11  7.678797e-12  8.175494e-12  2.545291e-12  6.713159e-13   
8   1.040352e-11  5.875515e-12  5.008560e-12  1.978441e-12  6.661468e-13   
9   1.466717e-11  6.518966e-12  5.301724e-12  2.766465e-12  6.291402e-13   
10  1.637913e-11  

## Classifier: ML Model

In [12]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


X = df_all[feature_cols].values
y = df_all['label'].values

#say you have 200 data points and have 10 bends. 20 dp in each bend. take one as a testing bend and the others as train.
# repeat this 20 times and average the results.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = make_pipeline(StandardScaler(), SVC(kernel='rbf',C=1.0, gamma='scale'))
scores = cross_val_score(clf, X, y, cv=cv)

print("=== Classification Results ===")
print(f"5-fold cross-validation accuracy: {scores.mean()*100:.1f}% +/- {scores.std()*100:.1f}%")
print(f"Individual fold scores: {[f'{s*100:.1f}%' for s in scores]}")
print()
print(f"Chance level: 50.0%")
print(f"Your result:  {scores.mean()*100:.1f}%")
if scores.mean() > 0.55:
    print("\nMeaningfully above chance -- the brain signal is detectable.")
else:
    print("\nClose to chance -- try more trials, different channels, or different bands.")

=== Classification Results ===
5-fold cross-validation accuracy: 66.7% +/- 0.0%
Individual fold scores: ['66.7%', '66.7%', '66.7%', '66.7%', '66.7%']

Chance level: 50.0%
Your result:  66.7%

Meaningfully above chance -- the brain signal is detectable.


## Feature Significance

In [13]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest gives interpretable feature importances
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#9B1B3D' if 'C3' in name else '#1E3A5F' if 'C4' in name else '#666666'
          for name in importances_sorted.index]
ax.barh(importances_sorted.index, importances_sorted.values, color=colors, alpha=0.85)
ax.set_xlabel('Feature importance', fontsize=10)
ax.set_title('Random Forest Feature Importance\n(red=C3, blue=C4, gray=Cz)',
             fontsize=11)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('eeg_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eeg_feature_importance.png")
print()
print("Top 5 most important features:")
print(importances_sorted.tail(5)[::-1])

Saved: eeg_feature_importance.png

Top 5 most important features:
Delta_C3    0.112411
Gamma_C3    0.093851
Beta_Cz     0.093697
Gamma_Cz    0.089649
Theta_Cz    0.085488
dtype: float64


C:\Users\borae\AppData\Local\Temp\ipykernel_23508\4072888749.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
import pickle

# Save feature matrix as CSV (for dashboard data display)
df_all.to_csv('eeg_features.csv', index=False)
print("Saved: eeg_features.csv")

# Save trained classifier
with open('eeg_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)
print("Saved: eeg_classifier.pkl")

# Save ERD matrix for the heatmap in the dashboard
np.save('eeg_erd_matrix.npy', erd_matrix)
print("Saved: eeg_erd_matrix.npy")

print()


Saved: eeg_features.csv
Saved: eeg_classifier.pkl
Saved: eeg_erd_matrix.npy

